# api_call_logger

Buffers Spotify API call records in memory and flushes them as a single Delta
batch append to `spotify_etl.raw.spotify_api_calls`.

Schema aligns with `notebooks/ddl/monitoring/ddl_api_monitoring.ipynb`.

Dependencies:
- `%run ../config/settings`

In [ ]:
# %run ../config/settings

In [ ]:
import json as _json
from pyspark.sql import functions as F

_API_CALLS_TABLE = f"{CATALOG}.{RAW_SCHEMA}.spotify_api_calls"
_buffer: list[dict] = []


def save_raw_api_call(
    run_id:               str,
    method:               str,
    endpoint:             str,
    request_url:          str,
    http_status:          int,
    payload_json:         str,
    req_hash:             str,
    request_params_json:  str = "{}",
    request_body_json:    str = "{}",
    response_headers_json: str = "{}",
    error:                str | None = None,
    attempt:              int = 1,
    token_name:           str | None = None,
) -> None:
    """Append one API call record to the in-memory buffer."""
    _buffer.append({
        "run_id":                run_id,
        "token_name":            token_name,
        "method":                method.upper(),
        "endpoint":              endpoint,
        "request_url":           request_url,
        "request_params_json":   request_params_json,
        "request_body_json":     request_body_json,
        "http_status":           http_status,
        "response_headers_json": response_headers_json,
        "payload_json":          payload_json,
        "error":                 error,
        "attempt":               attempt,
        "request_hash":          req_hash,
    })


def flush_api_call_buffer() -> int:
    """
    Write all buffered records to Delta (appending ingestion_ts and ingestion_date)
    and clear the buffer. Returns the number of rows written.
    """
    if not _buffer:
        return 0

    df = (
        spark.createDataFrame(_buffer)
        .withColumn("ingestion_ts",   F.current_timestamp())
        .withColumn("ingestion_date", F.current_date())
    )
    df.write.format("delta").mode("append").saveAsTable(_API_CALLS_TABLE)

    count = len(_buffer)
    _buffer.clear()
    return count